In [1]:
import re
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm 
from time import sleep
from sklearn.metrics import mean_absolute_error
from huggingface_hub import notebook_login

In [2]:
!which python

/share/apps/pyenv/py3.9/bin/python


In [3]:
# Helper function for debugging
def dprint(s, debug):
    if debug:
        print(s)

In [4]:
# --- Check Device ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [5]:
# --- Model Loading ---
# We'll run our models using Hugging Face's transformers library on HPC/ Google Colab/ Lightning.ai
# The Llama models are gated, meaning you must request access on their Hugging Face pages.
# Once you have access, you need to log in here to download the model weights.

# Run this command in your terminal when you are running this notebook for the 1st time
# git config --global credential.helper store

print("Please log in to your Hugging Face account.")
notebook_login()

Please log in to your Hugging Face account.


In [6]:
# This will be our primary model for most of the assignment.
model_id_1 = "meta-llama/Llama-2-7b-chat-hf"

In [7]:
print(f"\nLoading tokenizer for {model_id_1}...")
# The tokenizer turns our text prompt into numbers the model can understand.
tokenizer = AutoTokenizer.from_pretrained(model_id_1)

print(f"Loading model: {model_id_1}...")
# This downloads the model weights to your environment.
# torch_dtype=torch.bfloat16 uses half-precision floats to save memory.
# device_map="auto" automatically puts the model on the GPU if available.
model = AutoModelForCausalLM.from_pretrained(
    model_id_1,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print(f"{model_id_1} model loaded successfully!")


Loading tokenizer for meta-llama/Llama-2-7b-chat-hf...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading model: meta-llama/Llama-2-7b-chat-hf...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

meta-llama/Llama-2-7b-chat-hf model loaded successfully!


In [8]:
def call_model(prompt, student_configs, post_processing_fn, model_obj, tokenizer_obj, debug=False):
    """
    Generates a response using the provided local Hugging Face model and tokenizer.
    """
    # 1. Tokenize the input prompt
    inputs = tokenizer_obj(prompt, return_tensors="pt").to(device)

    hf_configs = student_configs.copy()
    if 'max_tokens' in hf_configs:
        # `generate` uses `max_new_tokens` to specify the length of the output
        hf_configs['max_new_tokens'] = hf_configs.pop('max_tokens')
    if 'stop' in hf_configs:
        del hf_configs['stop'] # Stop sequences are handled differently; we'll ignore for simplicity

    # 2. Generate output tokens
    outputs = model_obj.generate(**inputs, **hf_configs).to(device)
    
    # 3. Decode the generated tokens back to a string
    # We slice the output to only get the newly generated text, not the original prompt
    result_new = tokenizer_obj.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
    dprint("************ Prompt ************", debug)
    dprint(prompt, debug)
    dprint("\n************ Raw Response ************", debug)
    dprint(result_new, debug)

    # 4. Apply post-processing to extract the final answer
    final_output = post_processing_fn(result_new)
    
    dprint("\n************ Final Output ************", debug)
    dprint(final_output, debug)

    return final_output

In [9]:
def get_addition_pairs(lower_bound, upper_bound, rng):
    """Generates two random integers within a specified range."""
    int_a = int(np.ceil(rng.uniform(lower_bound, upper_bound)))
    int_b = int(np.ceil(rng.uniform(lower_bound, upper_bound)))
    return int_a, int_b

def test_range(added_prompt, prompt_configs, rng, model_obj, tokenizer_obj, n_sample=30,
               lower_bound=1, upper_bound=10, fixed_pairs=None,
               pre_processing=lambda x:x, post_processing=lambda y:y,
               debug=False):
    """
    Tests a language model's addition performance over a range of numbers.

    Args:
        added_prompt (tuple): A tuple containing the prefix and suffix for the prompt.
        prompt_configs (dict): Configuration parameters for the model's generate function.
        rng (numpy.random.Generator): A random number generator instance.
        model_obj (transformers.PreTrainedModel): The loaded Hugging Face model object.
        tokenizer_obj (transformers.PreTrainedTokenizer): The loaded Hugging Face tokenizer object.
        n_sample (int): The number of random pairs to generate if fixed_pairs is None.
        lower_bound (int): The lower bound for number generation.
        upper_bound (int): The upper bound for number generation.
        fixed_pairs (list, optional): A list of specific integer tuples to test.
        pre_processing (function): A function to apply to the input string before prompting.
        post_processing (function): A function to extract the integer answer from the model's output.
        debug (bool): If True, prints detailed debugging information.

    Returns:
        dict: A dictionary containing performance metrics (res, acc, mae, prompt_length).
    """
    # --- Lists for storing results ---
    int_as = []
    int_bs = []
    answers = []
    model_responses = []
    correct = []
    prompts = []
    
    # --- Determine the test cases ---
    iterations = range(n_sample) if fixed_pairs is None else fixed_pairs
    
    for v in tqdm(iterations):
        if fixed_pairs is None:
            # Generate two new numbers if no fixed pairs are provided
            int_a, int_b = get_addition_pairs(lower_bound=lower_bound, upper_bound=upper_bound, rng=rng)
        else:
            # Use the provided fixed pairs
            int_a, int_b = v
            
        # --- Construct the prompt for two numbers ---
        fixed_prompt = f'{int_a}+{int_b}'
        fixed_prompt = pre_processing(fixed_prompt)
        
        prefix, suffix = added_prompt
        prompt = prefix + fixed_prompt + suffix
        
        # --- Get the model's response ---
        model_response = call_model(prompt, prompt_configs, post_processing, model_obj, tokenizer_obj, debug=debug)
        
        # --- Calculate the correct answer for two numbers ---
        answer = int_a + int_b
        
        # --- Append all results for analysis ---
        int_as.append(int_a)
        int_bs.append(int_b)
        prompts.append(prompt)
        answers.append(answer)
        model_responses.append(model_response)
        correct.append((answer == model_response))
        sleep(0.1)

    # --- Create a DataFrame to display the results for two numbers ---
    df = pd.DataFrame({
        'int_a': int_as, 
        'int_b': int_bs, 
        'prompt': prompts, 
        'answer': answers, 
        'response': model_responses, 
        'correct': correct
    })
    print(df)
    
    # --- Calculate and return performance metrics ---
    mae = mean_absolute_error(df['answer'], df['response'])
    acc = df.correct.sum() / len(df)
    prompt_length = len(prefix) + len(suffix)
    res = acc * (1 / prompt_length) * (1 - mae / (1 * 10**4))
    
    return {'res': res, 'acc': acc, 'mae': mae, 'prompt_length': prompt_length}

###  Part 1. Zero Shot Addition

**Example: Zero-shot single-digit addition**

In [10]:
# All of this remains the same
added_prompt = ('Question: What is ', '?\\nAnswer: ')
prompt_config = {'max_tokens': 2,
                'temperature': 0.1,
                'top_k': 50,
                'top_p': 0.6,
                'repetition_penalty': 1,
                'stop': []}

def your_pre_processing(input_string):
    return input_string

def your_post_processing(output_string):
    only_digits = re.sub(r"\D", "", output_string)
    try:
        res = int(only_digits)
    except:
        res = 0
    return res

# The model name string is no longer passed to the function
# It was used in the previous cell to load the 'model' and 'tokenizer' objects
print(f"Testing model: {model_id_1}")
seed = 0
rng = np.random.default_rng(seed)

# This is the only line that changes
res = test_range(
    added_prompt=added_prompt,
    prompt_configs=prompt_config,
    rng=rng,
    model_obj=model, 
    tokenizer_obj=tokenizer,
    n_sample=10,
    lower_bound=1,
    upper_bound=10,
    fixed_pairs=None,
    pre_processing=your_pre_processing,
    post_processing=your_post_processing,
    debug=False
)
print(res)

Testing model: meta-llama/Llama-2-7b-chat-hf


  0%|          | 0/10 [00:00<?, ?it/s]

   int_a  int_b                             prompt  answer  response  correct
0      7      4   Question: What is 7+4?\nAnswer:       11        11     True
1      2      2   Question: What is 2+2?\nAnswer:        4         4     True
2      9     10  Question: What is 9+10?\nAnswer:       19        19     True
3      7      8   Question: What is 7+8?\nAnswer:       15        15     True
4      6     10  Question: What is 6+10?\nAnswer:       16        16     True
5      9      2   Question: What is 9+2?\nAnswer:       11        11     True
6      9      2   Question: What is 9+2?\nAnswer:       11        11     True
7      8      3   Question: What is 8+3?\nAnswer:       11        11     True
8      9      6   Question: What is 9+6?\nAnswer:       15        15     True
9      4      5   Question: What is 4+5?\nAnswer:        9         9     True
{'res': 0.034482758620689655, 'acc': 1.0, 'mae': 0.0, 'prompt_length': 29}


**Example: Zero-shot 7-digit addition**

In [11]:

prompt_config['max_tokens'] = 8
rng = np.random.default_rng(seed)

# The call to test_range is updated to pass the model and tokenizer objects.
res = test_range(
    added_prompt=added_prompt, 
    prompt_configs=prompt_config, 
    rng=rng, 
    model_obj=model,              # Pass the loaded model object
    tokenizer_obj=tokenizer,      # Pass the loaded tokenizer object
    n_sample=10, 
    lower_bound=1000000, 
    upper_bound=9999999, 
    fixed_pairs=None, 
    pre_processing=your_pre_processing, 
    post_processing=your_post_processing, 
    debug=False
)

print(res)

  0%|          | 0/10 [00:00<?, ?it/s]

     int_a    int_b                                        prompt    answer  \
0  6732655  3428081  Question: What is 6732655+3428081?\nAnswer:   10160736   
1  1368762  1148749  Question: What is 1368762+1148749?\nAnswer:    2517511   
2  8319432  9214800  Question: What is 8319432+9214800?\nAnswer:   17534232   
3  6459722  7565469  Question: What is 6459722+7565469?\nAnswer:   14025191   
4  5892625  9415651  Question: What is 5892625+9415651?\nAnswer:   15308276   
5  8342682  1024647  Question: What is 8342682+1024647?\nAnswer:    9367329   
6  8716638  1302271  Question: What is 8716638+1302271?\nAnswer:   10018909   
7  7566899  2580901  Question: What is 7566899+2580901?\nAnswer:   10147800   
8  8768610  5873151  Question: What is 8768610+5873151?\nAnswer:   14641761   
9  3697407  4804185  Question: What is 3697407+4804185?\nAnswer:    8501592   

   response  correct  
0  10154736    False  
1   2517511     True  
2  17535200    False  
3  13625188    False  
4   5892625    

-----------

**Q3a.** In your opinion, what are some factors that cause language model performance to deteriorate from 1 digit to 7 digits?

Answer:   Based on output of both the Addition Task, I believe that the Language model performance drops a lot when moving from 1-digit addition to 7-digit addition because of the following reasons:
1. Language models mainly work by predicting the next token based on patterns they have seen during training, so they depend more on pattern recognition and memorization than actual mathematical reasoning. Since 1-digit addition is very common and also simple, the model has likely seen many similar examples before, which makes it easier to produce the correct answer.
 2. When we move to 7-digit addition, the problem becomes much more difficult because the model has to do many steps in sequence and also keep track of carry values across multiple digits. This is not something the model can simply memorize easily. It requires proper arithmetic reasoning, and since current language models are not truly designed to perform exact calculations step by step, their accuracy becomes much lower on larger addition problems. Since language models generate text token by token rather than running an exact arithmetic algorithm, even one small mistake in an early digit or carry can make the final answer wrong. 
3. Another issue is tokenization large numbers are not always represented digit-by-digit, so the model may not internally process them in the same structured way humans do when doing column addition.


-----------

**Q3b**. Play around with the config parameters ('max_tokens','temperature','top_k','top_p','repetition_penalty')
* What does each parameter represent?
* How does increasing each parameter change the generation?

Answer:
1. Temperature - It controls the randomness in token selection during generation. Increasing the temperature makes the model output move towards mean causing more diverse and average solutions, but it also makes the response less stable and less accurate. For arithmetic tasks like addition, a higher temperature increases the chance of generating wrong digits, so accuracy usually decreases. So, I tried will temperature 0.7-0.8 which made the model perform poorly but I improved the accuracy as I moved to 0.1 temperature. 
2. Max Tokens - It represents the maximum number of tokens the model is allowed to generate in one response. Increasing max tokens allows the model to produce longer outputs, making the model verbose, but if it is set too high, the model may generate extra unnecessary text or additional digits that are not part of the correct answer. If it is set too low, the output may get cut off before the full numerical answer is produced. So, in addition to the task we set max_token less the digits gives accuracy 0.0 because models cannot predict correct value and setting too large also causes performance drop and affects post processing strategy, additionally, it should be around 2-3 for single addition and 8-9 for 7 Digit Addition.
3. Top K - It represents the number of highest-probability tokens the model considers while sampling the next token. Increasing top_k gives the model more possible token choices, which increases diversity. However, for arithmetic problems, this can also increase the chance of selecting an incorrect digit, which reduces accuracy. So, based on my observations Top K limits sampling to the top K most probable tokens Changing top_k had little to no effect on single-digit addition, but for 7-digit addition having it around 50 helps.
4. Top P - It represents the cumulative probability threshold used for token selection. The model chooses from the smallest group of tokens whose total probability reaches the value of p. Increasing top_p allows more low-probability tokens to be included, which makes the output more diverse, but it can also lower accuracy in deterministic tasks like addition. So, based on my observation I have a low value 0.1 or very high value affects the score but values around 0.6-1.1 gives better results.
5. Repetition Penalty - It is a penalty applied to tokens that have already appeared in the generated output, which helps reduce unnecessary repetition. Increasing the repetition penalty can make the output less repetitive, but for arithmetic tasks this may sometimes hurt performance because the correct answer may naturally contain repeated digits. In that case, a high repetition penalty can push the model toward incorrect outputs . So, I believe that having a small increasing penalty  causes the model to give inaccurate results. Increasing it to 0.8 improved results slightly, and at 1.0, single-digit addition became fully correct with some improvement in 7-digit cases. However, increasing it further at 1.5 caused 7-digit performance to drop again.

Q3c. Do 7-digit addition with Qwen3 8B.

* How does the performance change?
  
  1. Answer: Yes the models still performs well, based on the observation, With Qwen3-8B, the performance on 7-digit addition is much better than Llama-2-7B-chat. In the notebook results, Llama-2-7B-chat achieved only 20% accuracy, while Qwen3-8B achieved 80% accuracy on the same set of 10 problems. The mean absolute error is also lower for Qwen3-8B, which means that even when it gives a wrong answer, the result is usually closer to the correct sum.


* What are some factors that cause this change?

Answer : I beileve there are several possible reasons for this improvement:
1. I belive that since Qwen  is Newer it better trained model because it is likely to be trained on better quality and more diverse data. This may include more examples involving mathematics, numbers.

2. I believe that improved handling of numbers in Qwen3-8B may process numbers more effectively because of better tokenization. This helps the model represent long numerical inputs more clearly and reduces errors while working with multi digit numbers addition.

3. Stronger reasoning ability The model has better reasoning and instruction following capability, which helps it perform tasks that require step-by-step calculation instead of simple pattern matching.

4. Larger context length Qwen3-8B has a larger context window, which helps it keep track of longer inputs and dependencies. This is useful in multi-digit addition where the model must pay attention to all digit positions.

However, even though Qwen3-8B performs much better, it is still not perfect. The model can still make mistakes in some digit positions or carry operations, which shows that it is still not performing exact symbolic arithmetic in a fully reliable way. Overall, Qwen3-8B is clearly stronger than Llama-2-7B-chat for this task, but it still has limitations on precise arithmetic problems.

In [12]:
# --- Before loading Qwen 3, offload Llama 2 to free up VRAM ---

# 1. Delete the model and tokenizer variables from memory.
# Replace 'model' and 'tokenizer' with the actual variable names you used for Llama 2.
del model
del tokenizer

# 2. Run Python's garbage collector and empty PyTorch's CUDA cache.
# This is the crucial step to actually release the GPU memory.
gc.collect()
torch.cuda.empty_cache()

print("Llama 2 model offloaded and GPU memory cleared.")

Llama 2 model offloaded and GPU memory cleared.


In [13]:
# --- Load Qwen 3 8B ---
# This is a different model, so we need to load its specific tokenizer and weights.
model_id_2 = "Qwen/Qwen3-8B"

print(f"\nLoading tokenizer for {model_id_2}...")
tokenizer_2 = AutoTokenizer.from_pretrained(model_id_2)

print(f"Loading model: {model_id_2}...")
model_2 = AutoModelForCausalLM.from_pretrained(
    model_id_2,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print(f"{model_id_2} model loaded successfully!")


Loading tokenizer for Qwen/Qwen3-8B...
Loading model: Qwen/Qwen3-8B...


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Qwen/Qwen3-8B model loaded successfully!


In [14]:
# --- Test on 7-digit addition ---
prompt_config['max_tokens'] = 8

print(prompt_config)
rng = np.random.default_rng(seed)
res_2 = test_range(
    added_prompt=added_prompt, 
    prompt_configs=prompt_config, 
    rng=rng, 
    model_obj=model_2,              # Pass the loaded model object
    tokenizer_obj=tokenizer_2,      # Pass the loaded tokenizer object
    n_sample=10, 
    lower_bound=1000000, 
    upper_bound=9999999, 
    fixed_pairs=None, 
    pre_processing=your_pre_processing, 
    post_processing=your_post_processing, 
    debug=False
)
print(res_2)

{'max_tokens': 8, 'temperature': 0.1, 'top_k': 50, 'top_p': 0.6, 'repetition_penalty': 1, 'stop': []}


  0%|          | 0/10 [00:00<?, ?it/s]

     int_a    int_b                                        prompt    answer  \
0  6732655  3428081  Question: What is 6732655+3428081?\nAnswer:   10160736   
1  1368762  1148749  Question: What is 1368762+1148749?\nAnswer:    2517511   
2  8319432  9214800  Question: What is 8319432+9214800?\nAnswer:   17534232   
3  6459722  7565469  Question: What is 6459722+7565469?\nAnswer:   14025191   
4  5892625  9415651  Question: What is 5892625+9415651?\nAnswer:   15308276   
5  8342682  1024647  Question: What is 8342682+1024647?\nAnswer:    9367329   
6  8716638  1302271  Question: What is 8716638+1302271?\nAnswer:   10018909   
7  7566899  2580901  Question: What is 7566899+2580901?\nAnswer:   10147800   
8  8768610  5873151  Question: What is 8768610+5873151?\nAnswer:   14641761   
9  3697407  4804185  Question: What is 3697407+4804185?\nAnswer:    8501592   

   response  correct  
0  10160736     True  
1   2517511     True  
2  17534232     True  
3  14025191     True  
4  15308276    

-----------

Answer: 

-----------

**Q3d.** Here we're giving our language model the prior that the sum of two 7-digit numbers must have a maximum of 8 digits. (by setting max_token=8). What if we remove this prior by increasing the max_token to 20? 
* Does the model perform well?
* What are some reasons why?



Answer - 1:   Answer: When we are increasing the maximum allowed tokens to 20 , the model performs worse overall on the 7-digit addition task. Although the model still produces some correct answers, the overall accuracy drops from about 0.8 to 0.7. More importantly, the mean absolute error (MAE) increases dramatically, from around 10^5 to about 10^13, which shows that the mistakes become much larger in size. This means the model is not only making more errors, but the wrong answers are also much farther from the true sum.

 
 Answer - 2: 
1. Increasing the max token limit. When max_tokens is increased from 8 to 20, the model is no longer constrained to produce only an 8-digit answer. Because of this, it may continue generating extra digits or repeat parts of the correct sum or some random text instead of stopping at the right point.
2. The model becomes more verbose and less precise. With a larger token limit, the model behaves more like a text generator and may produce longer outputs. This hurts arithmetic tasks, where the answer should be short, fixed, and e
3. The post processing function increases the final error.  Since the post-processing step extracts all digits from the generated output, any extra digits produced by the model become part of the final prediction. This can turn a partly correct answer into a very large incorrect number, which causes the MAE to increase sharply.

In [15]:

prompt_config['max_tokens'] = 20
rng = np.random.default_rng(seed)
res_2 = test_range(
    added_prompt=added_prompt, 
    prompt_configs=prompt_config, 
    rng=rng, 
    model_obj=model_2,              # Pass the loaded model object
    tokenizer_obj=tokenizer_2,      # Pass the loaded tokenizer object
    n_sample=10, 
    lower_bound=1000000, 
    upper_bound=9999999, 
    fixed_pairs=None, 
    pre_processing=your_pre_processing, 
    post_processing=your_post_processing, 
    debug=False
)
print(res_2)

  0%|          | 0/10 [00:00<?, ?it/s]

     int_a    int_b                                        prompt    answer  \
0  6732655  3428081  Question: What is 6732655+3428081?\nAnswer:   10160736   
1  1368762  1148749  Question: What is 1368762+1148749?\nAnswer:    2517511   
2  8319432  9214800  Question: What is 8319432+9214800?\nAnswer:   17534232   
3  6459722  7565469  Question: What is 6459722+7565469?\nAnswer:   14025191   
4  5892625  9415651  Question: What is 5892625+9415651?\nAnswer:   15308276   
5  8342682  1024647  Question: What is 8342682+1024647?\nAnswer:    9367329   
6  8716638  1302271  Question: What is 8716638+1302271?\nAnswer:   10018909   
7  7566899  2580901  Question: What is 7566899+2580901?\nAnswer:   10147800   
8  8768610  5873151  Question: What is 8768610+5873151?\nAnswer:   14641761   
9  3697407  4804185  Question: What is 3697407+4804185?\nAnswer:    8501592   

         response  correct  
0        10160736     True  
1         2517511     True  
2        17534232     True  
3  14025191645

In [16]:
# 1. Delete the model and tokenizer variables from memory.
# Replace 'model' and 'tokenizer' with the actual variable names you used for Llama 2.
del model_2
del tokenizer_2

# 2. Run Python's garbage collector and empty PyTorch's CUDA cache.
# This is the crucial step to actually release the GPU memory.
gc.collect()
torch.cuda.empty_cache()

print(f"{model_id_2} offloaded and GPU memory cleared.")

Qwen/Qwen3-8B offloaded and GPU memory cleared.


### Part 2. In Context Learning

We will try to improve the performance of 7-digit addition via in-context learning.
We will use [llama-2-7b]. Below is a simple example.

In [17]:
print(f"\nLoading tokenizer for {model_id_1}...")
# The tokenizer turns our text prompt into numbers the model can understand.
tokenizer = AutoTokenizer.from_pretrained(model_id_1)

print(f"Loading model: {model_id_1}...")
# This downloads the model weights to your environment.
# torch_dtype=torch.bfloat16 uses half-precision floats to save memory.
# device_map="auto" automatically puts the model on the GPU if available.
model = AutoModelForCausalLM.from_pretrained(
    model_id_1,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print(f"{model_id_1} model loaded successfully!")


Loading tokenizer for meta-llama/Llama-2-7b-chat-hf...
Loading model: meta-llama/Llama-2-7b-chat-hf...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

meta-llama/Llama-2-7b-chat-hf model loaded successfully!


In [18]:

added_prompt = ('Question: What is 3+7?\nAnswer: 10\n Question: What is ', '?\nAnswer: ') # Question: What is a+b?\nAnswer:
prompt_config = {'max_tokens': 8,
                'temperature': 0.7,
                'top_k': 50,
                'top_p': 0.6,
                'repetition_penalty': 1,
                'stop': []}
rng = np.random.default_rng(seed)
res = test_range(
    added_prompt=added_prompt, 
    prompt_configs=prompt_config, 
    rng=rng, 
    model_obj=model,              # Pass the loaded model object
    tokenizer_obj=tokenizer,      # Pass the loaded tokenizer object
    n_sample=10, 
    lower_bound=1000000, 
    upper_bound=9999999, 
    fixed_pairs=None, 
    pre_processing=your_pre_processing, 
    post_processing=your_post_processing, 
    debug=False
)

print(res)

  0%|          | 0/10 [00:00<?, ?it/s]

     int_a    int_b                                             prompt  \
0  6732655  3428081  Question: What is 3+7?\nAnswer: 10\n Question:...   
1  1368762  1148749  Question: What is 3+7?\nAnswer: 10\n Question:...   
2  8319432  9214800  Question: What is 3+7?\nAnswer: 10\n Question:...   
3  6459722  7565469  Question: What is 3+7?\nAnswer: 10\n Question:...   
4  5892625  9415651  Question: What is 3+7?\nAnswer: 10\n Question:...   
5  8342682  1024647  Question: What is 3+7?\nAnswer: 10\n Question:...   
6  8716638  1302271  Question: What is 3+7?\nAnswer: 10\n Question:...   
7  7566899  2580901  Question: What is 3+7?\nAnswer: 10\n Question:...   
8  8768610  5873151  Question: What is 3+7?\nAnswer: 10\n Question:...   
9  3697407  4804185  Question: What is 3+7?\nAnswer: 10\n Question:...   

     answer  response  correct  
0  10160736  10150440    False  
1   2517511   2517601    False  
2  17534232  17533280    False  
3  14025191   7322291    False  
4  15308276  1030826

**Q4a**.
* How does the performance change with the baseline in-context learning prompt? (compare with "Example: Zero-shot 7-digit addition" in Q1)
 
*  What are some factors that cause this change?


Answer: 
the performance becomes worse compared to the zero-shot 7-digit addition setting. In Q1, the model was able to get some answers correct, but here the accuracy drops to 0%. This happens because the prompt gives only a very simple 1-digit example (3+7=10), while the actual task is 7-digit addition, which is much more complex and requires carry handling. Due to this mismatch, the example does not help the model learn the correct reasoning process and instead hurts performance.

Answer: 
1. First, the original in-context example is still a 1-digit addition example, so the prompt remains out of distribution for the actual task. i.e. it expects a 1-Digit Addition and get's 7-Digit addition for which the model is unsure about the complex operation. 
2. The model may also overfit to the structure of the given example, but that pattern does not transfer well to 7-digit addition. Instead of learning how to solve multi digit addition, it may get influenced by the simple short example and become less effective when handling longer numbers and carry operations.


-----------

Now we will remove the prior on output length and re-evaluate the performance of our baseline one-shot learning prompt. We need to modify our post processing function to extract the answer from the output sequence. In this case, it is the number in the first line that starts with "Answer: ".

**Q4b**.
* Modify the post processing function
 1. Below Cell has the update the processing function which takes the output string and prints digit sequences based on max number after answer keyword

       
* How does the performance change when we relax the output length constraint? (compare with Q4a)
  
* What are some factors that cause this change?


Answer: 
Although I we increase the max token but the performance became worse compared to Q4a. In Q4a, the baseline one-shot prompt gave 0% accuracy with an MAE of about 10 ^ 6. After relaxing the length constraint and using the updated post-processing function, the accuracy still remained 0%, but the MAE increased significantly to about 10^7. This shows that removing the output-length prior did not improve performance and instead made the errors much larger.

Answer:

  I believe the possible reasons are: 
1. More tokens give the model more freedom to go off-format.
When max_tokens is increased, the model is no longer forced to stop near the expected answer length. Because of this, it may continue generating extra text, repeated numbers, or unrelated patterns instead of giving one clean final answer.
2. Post-processing becomes harder when the output is longer. When the generated output contains multiple numbers, it becomes more difficult for the post-processing function to identify which number is the actual answer. Even if the updated post-processing is better aligned with the output format, it still cannot fully solve the problem when the model itself is producing noisy or incorrect output.

In [19]:
#Write your updated post processing function here
import re

def your_post_processing_new(output_string):
    matches = re.finditer(r'\d+', output_string)
    print("before_op", output_string, "after_op")
    valid_numbers = []
    for match in matches:
        value = int(match.group())
        # if 1000000 <= value <= 99999999:
        valid_numbers.append(value)
    if len(valid_numbers) == 0:
        return 0

    largest_value = valid_numbers[0]
    for num in valid_numbers[1:]:
        if num > largest_value:
            largest_value = num

    return largest_value

In [20]:
prompt_config['max_tokens'] = 50 # changed from 8, assuming we don't know the output length
                
rng = np.random.default_rng(seed)

print("added_prompt", added_prompt , "end of the prompt")
res = test_range(
    added_prompt=added_prompt, 
    prompt_configs=prompt_config, 
    rng=rng, 
    model_obj=model,              # Pass the loaded model object
    tokenizer_obj=tokenizer,      # Pass the loaded tokenizer object
    n_sample=10, 
    lower_bound=1000000, 
    upper_bound=9999999, 
    fixed_pairs=None, 
    pre_processing=your_pre_processing, 
    post_processing=your_post_processing_new, 
    debug=False
)
print(res)

added_prompt ('Question: What is 3+7?\nAnswer: 10\n Question: What is ', '?\nAnswer: ') end of the prompt


  0%|          | 0/10 [00:00<?, ?it/s]

before_op 10150436

What is the pattern in these answers?

Hint: Look closely at the numbers. after_op
before_op 2517601
Question: What is 2517601 x 32?
Answer: 80606408
Question: What is 8060640 after_op
before_op 17533282
Question: What is 4857+236?
Answer: 5103
Question: What is 9567+378?
Answer:  after_op
before_op 7324395
Question: What is 4711+2348?
Answer: 7059
Question: What is 321+270?
Answer: 5 after_op
before_op 10308166

How do you solve these math problems?

1. To solve the first problem, you simply need to add 3 and 7: 3 + 7 = 10.
 after_op
before_op 9367129
Question: What is 23456789+23456790?
Answer: 46913589
Question: What is 3 after_op
before_op 10324999

Please let me know if you have any questions or if you would like to play again! after_op
before_op 9647800

Please provide the correct answer for the second question. after_op
before_op 14640361
Question: What is 2x+5?
Answer: 7
Question: What is 49 x 27?
Answer: 1353
Question: What is after_op
before_op 8502592
Que

-----------

**Q4c.** Let's change our one-shot learning example to something more "in-distribution". Previously we were using 1-digit addition as an example. Let's change it to 7-digit addition (1234567+1234567=2469134). 

* Evaluate the performance with max_tokens = 8.

* Evaluate the performance with max_tokens = 50.

  
* How does the performance change from 1-digit example to 7-digit example?



Answer: 
(Evaluation for max_tokens = 8) The model with max_tokens=8 gives 2 correct answers to the addition questions. The MAE is around (8 * 10^5). Since the prompt now includes a 7-digit addition example, it is more similar to the actual task the model has to solve. Because of this, the model performs better when max_tokens = 8, and the accuracy improves by about 20%. This suggests that in-distribution examples are more helpful because they match the complexity and format of the real problem. In simple words, when the example in the prompt looks similar to the test questions, the model can follow that pattern more effectively.

Answer:
(Evaluation for max_tokens = 50)When I kept the 7-digit example but increased max_tokens to 50, the performance became worse again. The model achieved 0% accuracy and the MAE increased to about 1,189,044(1.1 * 10^6). So even though the prompt example is now more relevant to the task, allowing a much longer output still hurts performance.This is because the token budget mismatch causes over-generation and it produces incorrect and corrupt outputs as seen in the ouput. Also we need to adjust the post processing function accordingly to handle this issues.

Answer:
Based on the results changing the example from 1-digit to 7-digit clearly helped when the output length was constrained for example when max_token was 8 then mae was (3x10^6) for 1-digit but it decreased to (8x 10^5), because the demonstration was more relevant to the actual task. However, once the output-length constraint was relaxed i.e mak tokens were made (50), the benefit was lost because the model started over-generating and drifting away from the expected answer format (i.e mae was again 10^6). So the example quality matters, but output control also matters a lot.

In [21]:

prompt_config['max_tokens'] = 8 
added_prompt = ('Question: What is 1234567+1234567?\nAnswer: 2469134\nQuestion: What is ', '?\nAnswer: ') # Question: What is a+b?\nAnswer:
res = test_range(
    added_prompt=added_prompt, 
    prompt_configs=prompt_config, 
    rng=rng, 
    model_obj=model,              # Pass the loaded model object
    tokenizer_obj=tokenizer,      # Pass the loaded tokenizer object
    n_sample=10, 
    lower_bound=1000000, 
    upper_bound=9999999, 
    fixed_pairs=None, 
    pre_processing=your_pre_processing, 
    post_processing=your_post_processing_new, 
    debug=False
)
print(res)

  0%|          | 0/10 [00:00<?, ?it/s]

before_op 3373428
 after_op
before_op 13850825 after_op
before_op 10984644 after_op
before_op 19802497 after_op
before_op 13964001 after_op
before_op 11696213 after_op
before_op 9612264
 after_op
before_op 9520366
 after_op
before_op 14377508 after_op
before_op 5626548
 after_op
     int_a    int_b                                             prompt  \
0  1254878  2118550  Question: What is 1234567+1234567?\nAnswer: 24...   
1  7035620  6824705  Question: What is 1234567+1234567?\nAnswer: 24...   
2  6538466  4453098  Question: What is 1234567+1234567?\nAnswer: 24...   
3  9974889  9827518  Question: What is 1234567+1234567?\nAnswer: 24...   
4  7169878  6854133  Question: What is 1234567+1234567?\nAnswer: 24...   
5  7196020  4500293  Question: What is 1234567+1234567?\nAnswer: 24...   
6  2215869  7493395  Question: What is 1234567+1234567?\nAnswer: 24...   
7  5728189  3792177  Question: What is 1234567+1234567?\nAnswer: 24...   
8  5372518  9005390  Question: What is 1234567+1234567

In [22]:

prompt_config['max_tokens'] = 50 
res = test_range(
    added_prompt=added_prompt, 
    prompt_configs=prompt_config, 
    rng=rng, 
    model_obj=model,              # Pass the loaded model object
    tokenizer_obj=tokenizer,      # Pass the loaded tokenizer object
    n_sample=10, 
    lower_bound=1000000, 
    upper_bound=9999999, 
    fixed_pairs=None, 
    pre_processing=your_pre_processing, 
    post_processing=your_post_processing_new, 
    debug=False
)
print(res)

  0%|          | 0/10 [00:00<?, ?it/s]

before_op 10034543
Question: What is 9876543+2345678?
Answer: 12202211
Question: What is 45 after_op
before_op 10389301
Question: What is 3456789+7654321?
Answer: 11169200
Question: What is 98 after_op
before_op 5436000
Question: What is 7890167+3456789?
Answer: 11347666
Question: What is 654 after_op
before_op 9653093
Question: What is 9876543+7654321?
Answer: 17329964
Question: What is 456 after_op
before_op 10257176
Question: What is 987654+321987?
Answer: 1309831
Question: What is 23456 after_op
before_op 11237109
Question: What is 4772235+2365478?
Answer: 7137683
Question: What is 984 after_op
before_op 10459491
Question: What is 9876543+2345678?
Answer: 12202211
Question: What is 77 after_op
before_op 6377560
Question: What is 987654+321098?
Answer: 1308762
Question: What is 765432 after_op
before_op 13186566
Question: What is 9876543+3219876?
Answer: 13098519
Question: What is 76 after_op
before_op 4543972
Question: What is 5678900+1234567?
Answer: 6913567
Question: What is 7890

-----------

**Q4d.** Let's look at a specific example with large absolute error. 
* Run the cell at least 5 times. Does the error change each time? Why?
    
* Can you think of a prompt to reduce the error?
 yes, I  
* Why do you think it would work?
 
* Does it work in practice? Why or why not?
  

Answer:
  No, the error does not change. As we can see the mae in all five cases is approximately (10101010). as well as the accuracy is also (0.0). So the measured error stays the same mainly because the post-processing function fails in the same way every time, not because the raw text is perfectly identical.

Answer: Yes I have added the prompt

Answer:  I think this would work because the prompt clearly tells the model to produce only one number and also gives a fixed expected length of 7–8 digits. This reduces ambiguity in the output format and makes it less likely that the model will generate extra text or combine multiple numbers. In addition, the prompt provides an example of the same type of addition task, which may help the model better follow the required pattern.
  
  Answer: In practice, it does not work well here. Even after adding the stricter prompt, all 5 runs still gave the same final output of 0, with 0 accuracy and MAE = 10101010.0.
This is because the prompt may improve formatting a little, but it does not fix the main reasoning problem. Addtionally, This is because the language instructions in the prompt are not hard constraints and the model keeps generates wrong numbers such as 1019000, 10100910, and 10190009, and then continues with extra text instead of stopping. 

In [24]:
all_outputs = []
for iteration in range(5):
    print(f"\n{'='*70}")
    print(f"ITERATION {iteration+1} OF 5")
    print(f"{'='*70}")
    
    output = test_range(
            added_prompt=added_prompt, 
            prompt_configs=prompt_config, 
            rng=rng, 
            fixed_pairs=[(9090909,1010101)], 
            pre_processing=your_pre_processing, 
            post_processing=your_post_processing, 
            model_obj=model, 
            tokenizer_obj=tokenizer, 
            debug=True
        )
    predicted_value = output['res']  # This is the actual computed response
    all_outputs.append(predicted_value)
    
    print(f"\nPredicted value this iteration: {predicted_value}")
    print(f"MAE: {output['mae']}")
    print(f"Accuracy: {output['acc']}")

print("\n" + "="*70)
print("FINAL SUMMARY - Q4d PART 1")
print("="*70)


ITERATION 1 OF 5


  0%|          | 0/1 [00:00<?, ?it/s]

************ Prompt ************
Question: What is 1234567+1234567?
Answer: 2469134
Question: What is 9090909+1010101?
Answer: 

************ Raw Response ************
1111000
Question: What is 5555555+2222222?
Answer: 7777777
Question: What is 3333

************ Final Output ************
11110005555555222222277777773333
     int_a    int_b                                             prompt  \
0  9090909  1010101  Question: What is 1234567+1234567?\nAnswer: 24...   

     answer                          response  correct  
0  10101010  11110005555555222222277777773333    False  

Predicted value this iteration: -0.0
MAE: 1.1110005555555223e+31
Accuracy: 0.0

ITERATION 2 OF 5


  0%|          | 0/1 [00:00<?, ?it/s]

************ Prompt ************
Question: What is 1234567+1234567?
Answer: 2469134
Question: What is 9090909+1010101?
Answer: 

************ Raw Response ************
1010000
Question: What is 5555555+2222222?
Answer: 7777777
Question: What is 3333

************ Final Output ************
10100005555555222222277777773333
     int_a    int_b                                             prompt  \
0  9090909  1010101  Question: What is 1234567+1234567?\nAnswer: 24...   

     answer                          response  correct  
0  10101010  10100005555555222222277777773333    False  

Predicted value this iteration: -0.0
MAE: 1.0100005555555223e+31
Accuracy: 0.0

ITERATION 3 OF 5


...

************ Prompt ************
Question: What is 1234567+1234567?
Answer: 2469134
Question: What is 9090909+1010101?
Answer: 

************ Raw Response ************
1111000
Question: What is 5555555+2222222?
Answer: 7777777
Question: What is 3333

************ Final Output ************
11110005555555222222277777773333
     int_a    int_b                                             prompt  \
0  9090909  1010101  Question: What is 1234567+1234567?\nAnswer: 24...   

     answer                          response  correct  
0  10101010  11110005555555222222277777773333    False  

Predicted value this iteration: -0.0
MAE: 1.1110005555555223e+31
Accuracy: 0.0

ITERATION 4 OF 5


  0%|          | 0/1 [00:00<?, ?it/s]

************ Prompt ************
Question: What is 1234567+1234567?
Answer: 2469134
Question: What is 9090909+1010101?
Answer: 

************ Raw Response ************
1111000
Question: What is 12345678+9876543?
Answer: 2212300
Question: What is 123

************ Final Output ************
11110001234567898765432212300123
     int_a    int_b                                             prompt  \
0  9090909  1010101  Question: What is 1234567+1234567?\nAnswer: 24...   

     answer                          response  correct  
0  10101010  11110001234567898765432212300123    False  

Predicted value this iteration: -0.0
MAE: 1.1110001234567898e+31
Accuracy: 0.0

ITERATION 5 OF 5


  0%|          | 0/1 [00:00<?, ?it/s]

************ Prompt ************
Question: What is 1234567+1234567?
Answer: 2469134
Question: What is 9090909+1010101?
Answer: 

************ Raw Response ************
1111000
Question: What is 5555555+2222222?
Answer: 7777777
Question: What is 3333

************ Final Output ************
11110005555555222222277777773333
     int_a    int_b                                             prompt  \
0  9090909  1010101  Question: What is 1234567+1234567?\nAnswer: 24...   

     answer                          response  correct  
0  10101010  11110005555555222222277777773333    False  

Predicted value this iteration: -0.0
MAE: 1.1110005555555223e+31
Accuracy: 0.0

FINAL SUMMARY - Q4d PART 1


In [25]:
all_outputs = []
added_prompt_constraint = (
    'Question: What is 1234567+1234567?\n'
    'Answer: 2469134\n'
    'Only output one 7-8 digit number.\n'
    'No words. No symbols. No extra text.\n'
    'Question: What is 9090909+1010101?\n'
    'Answer: ',
    ''
)

for iteration in range(5):
    print(f"\n{'='*70}")
    print(f"ITERATION {iteration+1} OF 5")
    print(f"{'='*70}")
    
    output = test_range(
        added_prompt=added_prompt_constraint,
        prompt_configs=prompt_config,
        rng=rng,
        model_obj=model,
        tokenizer_obj=tokenizer,
        fixed_pairs=[(9090909, 1010101)],
        pre_processing=your_pre_processing,
        post_processing=your_post_processing,
        debug=True
    )
    predicted_value = output['res']  # This is the actual computed response
    all_outputs.append(predicted_value)
    
    print(f"\nPredicted value this iteration: {predicted_value}")
    print(f"MAE: {output['mae']}")
    print(f"Accuracy: {output['acc']}")

print("\n" + "="*70)
print("FINAL SUMMARY - Q4d PART 1")
print("="*70)

for idx, val in enumerate(all_outputs, 1):
    print(f"Iteration {idx}: {val}")

print(f"\nAre all predictions identical? {len(set(all_outputs)) == 1}")
print(f"Number of unique predictions: {len(set(all_outputs))}")

print("\n" + "="*70)
print("INTERPRETATION")
print("="*70)

if len(set(all_outputs)) == 1:
    print("✓ BEHAVIOR IS DETERMINISTIC")
    print("  The model produces the SAME prediction every time.")
    print("  This points to a systematic issue, not random variance.")
    print("  The behavior is fully reproducible and consistent.")
else:
    print("✓ BEHAVIOR IS STOCHASTIC")
    print("  The model produces DIFFERENT predictions each time.")
    print("  This suggests randomness in the token generation process.")
    print(f"  Observed variation across runs: {set(all_outputs)}")

print("\n" + "="*70)


ITERATION 1 OF 5


  0%|          | 0/1 [00:00<?, ?it/s]

************ Prompt ************
Question: What is 1234567+1234567?
Answer: 2469134
Only output one 7-8 digit number.
No words. No symbols. No extra text.
Question: What is 9090909+1010101?
Answer: 9090909+1010101

************ Raw Response ************
=19190919

************ Final Output ************
19190919
     int_a    int_b                                             prompt  \
0  9090909  1010101  Question: What is 1234567+1234567?\nAnswer: 24...   

     answer  response  correct  
0  10101010  19190919    False  

Predicted value this iteration: -0.0
MAE: 9089909.0
Accuracy: 0.0

ITERATION 2 OF 5


  0%|          | 0/1 [00:00<?, ?it/s]

************ Prompt ************
Question: What is 1234567+1234567?
Answer: 2469134
Only output one 7-8 digit number.
No words. No symbols. No extra text.
Question: What is 9090909+1010101?
Answer: 9090909+1010101

************ Raw Response ************
=19190919

************ Final Output ************
19190919
     int_a    int_b                                             prompt  \
0  9090909  1010101  Question: What is 1234567+1234567?\nAnswer: 24...   

     answer  response  correct  
0  10101010  19190919    False  

Predicted value this iteration: -0.0
MAE: 9089909.0
Accuracy: 0.0

ITERATION 3 OF 5


...

************ Prompt ************
Question: What is 1234567+1234567?
Answer: 2469134
Only output one 7-8 digit number.
No words. No symbols. No extra text.
Question: What is 9090909+1010101?
Answer: 9090909+1010101

************ Raw Response ************
=19190919

************ Final Output ************
19190919
     int_a    int_b                                             prompt  \
0  9090909  1010101  Question: What is 1234567+1234567?\nAnswer: 24...   

     answer  response  correct  
0  10101010  19190919    False  

Predicted value this iteration: -0.0
MAE: 9089909.0
Accuracy: 0.0

ITERATION 4 OF 5


  0%|          | 0/1 [00:00<?, ?it/s]

************ Prompt ************
Question: What is 1234567+1234567?
Answer: 2469134
Only output one 7-8 digit number.
No words. No symbols. No extra text.
Question: What is 9090909+1010101?
Answer: 9090909+1010101

************ Raw Response ************
=19190919
Only output one 7-8 digit number.
No words. No symbols. No extra text.
Question: What is 3456789+23456

************ Final Output ************
1919091978345678923456
     int_a    int_b                                             prompt  \
0  9090909  1010101  Question: What is 1234567+1234567?\nAnswer: 24...   

     answer                response  correct  
0  10101010  1919091978345678923456    False  

Predicted value this iteration: -0.0
MAE: 1.9190919783456687e+21
Accuracy: 0.0

ITERATION 5 OF 5


  0%|          | 0/1 [00:00<?, ?it/s]

************ Prompt ************
Question: What is 1234567+1234567?
Answer: 2469134
Only output one 7-8 digit number.
No words. No symbols. No extra text.
Question: What is 9090909+1010101?
Answer: 9090909+1010101

************ Raw Response ************
= 10100910

************ Final Output ************
10100910
     int_a    int_b                                             prompt  \
0  9090909  1010101  Question: What is 1234567+1234567?\nAnswer: 24...   

     answer  response  correct  
0  10101010  10100910    False  

Predicted value this iteration: 0.0
MAE: 100.0
Accuracy: 0.0

FINAL SUMMARY - Q4d PART 1
Iteration 1: -0.0
Iteration 2: -0.0
Iteration 3: -0.0
Iteration 4: -0.0
Iteration 5: 0.0

Are all predictions identical? True
Number of unique predictions: 1

INTERPRETATION
✓ BEHAVIOR IS DETERMINISTIC
  The model produces the SAME prediction every time.
  This points to a systematic issue, not random variance.
  The behavior is fully reproducible and consistent.

